# Tutorial 2 — Fine-tuning a pretrained transformer for clickbait detection

**Goal:** fine-tune a pretrained encoder (XLM-RoBERTa) end-to-end with the Hugging Face `Trainer`, evaluate it, and test it on a language it never saw labels for.

**Data:** English clickbait headlines (Chakraborty et al., 2016) and Indonesian CLICK-ID headlines (William & Sari, 2020), both from the Hugging Face Hub.

**Runtime:** needs a GPU — *Runtime → Change runtime type → T4 GPU*. Training takes a few minutes.

In [ ]:
!pip -q install -U transformers datasets accelerate evaluate scikit-learn

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. Load data and make train / validation / test splits

In [ ]:
from datasets import load_dataset

ds = load_dataset("marksverdhei/clickbait_title_classification", split="train")
ds = ds.rename_columns({"title": "text", "clickbait": "label"})
ds = ds.class_encode_column("label")          # needed for stratified splitting

split = ds.train_test_split(test_size=0.2, stratify_by_column="label", seed=42)
trainval = split["train"].train_test_split(test_size=0.1, stratify_by_column="label", seed=42)
train_ds, val_ds, test_ds = trainval["train"], trainval["test"], split["test"]
print(len(train_ds), len(val_ds), len(test_ds))
print(train_ds[0])

**Why three splits?** We pick settings by looking at the validation set and touch the test set **once**, at the end. Tuning on the test set inflates results.

## 2. Tokenize with the model's own tokenizer

The model only understands the token IDs it was pretrained with, so we must use its tokenizer. Headlines are short, so 48 tokens is enough.

In [ ]:
from transformers import AutoTokenizer

MODEL = "xlm-roberta-base"   # multilingual; alternatives: "roberta-base" (EN), "allegro/herbert-base-cased" (PL), "indobenchmark/indobert-base-p1" (ID)
tok = AutoTokenizer.from_pretrained(MODEL)

example = "You won't believe what happened next"
print(tok.tokenize(example))
print(tok(example)["input_ids"])

In [ ]:
def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=48)

train_tok = train_ds.map(tokenize, batched=True)
val_tok = val_ds.map(tokenize, batched=True)
test_tok = test_ds.map(tokenize, batched=True)

## 3. Model, metrics and training

In [ ]:
import numpy as np
from scipy.special import softmax
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2,
    id2label={0: "not clickbait", 1: "clickbait"},
    label2id={"not clickbait": 0, "clickbait": 1})

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=-1)[:, 1]       # AUC from probabilities, not hard labels
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro"),
            "roc_auc": roc_auc_score(labels, probs)}

args = TrainingArguments(
    output_dir="clickbait-xlmr",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_tok, eval_dataset=val_tok,
                  data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=compute_metrics)
trainer.train()

## 4. Final evaluation on the test set (once)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

out = trainer.predict(test_tok)
print(out.metrics)
preds = out.predictions.argmax(-1)
print(classification_report(out.label_ids, preds, target_names=["not clickbait", "clickbait"], digits=4))
print("Confusion matrix:\n", confusion_matrix(out.label_ids, preds))

Compare with Tutorial 1. Is fine-tuning worth the extra GPU time compared with frozen embeddings + logistic regression?

## 5. Use the model and measure inference time

In [ ]:
import time
from transformers import pipeline

clf = pipeline("text-classification", model=trainer.model, tokenizer=tok,
               device=0 if torch.cuda.is_available() else -1)
mine = ["You won't believe what this professor said in class today",
        "European Parliament approves new rules on artificial intelligence",
        "Nie uwierzysz, co znaleziono w Wiśle!",
        "Ini alasannya kenapa kamu harus minum air putih"]
for t, r in zip(mine, clf(mine)):
    print(f"{r['label']:>14}  {r['score']:.2f}  {t}")

sample = test_ds["text"][:1000]
t0 = time.time(); clf(sample, batch_size=64); dt = time.time() - t0
print(f"Inference: {dt/len(sample)*1000:.2f} ms per headline (batched, {'GPU' if torch.cuda.is_available() else 'CPU'})")

## 6. Zero-shot on Indonesian (CLICK-ID)

The model was fine-tuned **only on English labels**. XLM-R was pretrained on 100 languages — does the clickbait knowledge transfer?

The cell first prints examples of each label so you can **check which number means clickbait** — always verify label conventions of a dataset you did not build.

In [ ]:
idd = load_dataset("mteb/IndonesianIdClickbaitClassification", split="train").to_pandas()
for lab in sorted(idd.label.unique()):
    print(f"--- label {lab} ---")
    for t in idd[idd.label == lab].text.sample(4, random_state=0):
        print("  ", t)

In [ ]:
ID_CLICKBAIT = 1   # change this if the examples above show the opposite convention

y_id = (idd["label"].values == ID_CLICKBAIT).astype(int)
res = clf(idd["text"].tolist(), batch_size=64)
p_id = np.array([r["score"] if r["label"] == "clickbait" else 1 - r["score"] for r in res])
pred_id = (p_id >= 0.5).astype(int)
print(classification_report(y_id, pred_id, target_names=["not clickbait", "clickbait"], digits=4))
print("ROC-AUC:", round(roc_auc_score(y_id, p_id), 4))

**Discuss:** look at precision vs recall for the clickbait class. If ROC-AUC is reasonable but recall is low, the model *ranks* Indonesian headlines sensibly but its scores are shifted — Tutorial 3 shows how to fix that cheaply.

## Exercises

1. Swap `MODEL` for `roberta-base` (English-only). Better on English? What happens on Indonesian?
2. Train for 1 and 3 epochs. Watch validation loss — when does it start overfitting?
3. Train with only 10% of the English training data. How much does F1 drop compared with the embedding baseline at 10%?

---
*Data:* Chakraborty et al. (2016), *Stop Clickbait*, ASONAM · William, A. & Sari, Y. (2020), *CLICK-ID: A novel dataset for Indonesian clickbait headlines*, Data in Brief 32, 106231 (MTEB copy, CC-BY-4.0).